# Dag 4: AutoML

**Nøglebegreber:** `automl.classification`, `automl.regression`, `featurization`, `primary_metric`, `AutoMLJob`

**Læringsmål:**
- Forstå hvad AutoML er og hvornår det er nyttigt
- Konfigurere og submitte et AutoML klassifikationsjob via Azure ML SDK v2
- Styre `primary_metric` og forstå hvilke metrics der er tilgængelige per opgavetype
- Konfigurere `featurization` og forstå hvad det gør automatisk
- Sætte exit-kriterier og limits for at styre ressourceforbrug
- Hente det bedste AutoML-run og dets model
- Forstå forskellen på AutoML klassifikation vs. regression i konfigurationen

**Forudsætninger:** Dag 1 (data assets), Dag 2 (environments + command jobs), Dag 3 (MLflow logging)

## 1. MLClient
Opret forbindelse til workspace

In [1]:
import sys
sys.path.append("..")

from src.utils import init_ml_client
ml_client = init_ml_client()

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


## 2. Hvad er AutoML?

AutoML (Automated Machine Learning) automatiserer valget af algoritme, feature engineering og hyperparameter-tuning. Azure ML's AutoML prøver systematisk en lang række modeller og preprocessing-kombinationer og returnerer den bedste baseret på en valgt metric.

AutoML understøtter tre primære opgavetyper:
- **Klassifikation** — forudsig en kategorisk label (f.eks. om en medarbejder forlader virksomheden)
- **Regression** — forudsig en kontinuert værdi (f.eks. månedlig indkomst)
- **Tidsserier** — forudsig fremtidige værdier baseret på historiske data

> **Eksamen tip:** I SDK v2 bruger du `azure.ai.ml.automl` modulet med funktionerne `automl.classification()`, `automl.regression()` osv. AutoML-jobs er **ikke** `command()` jobs — de er egne job-typer der submittes med `ml_client.jobs.create_or_update()` på samme måde, men returnerer en `AutoMLJob`-instans.

**Opgave:** List de tilgængelige compute-targets i dit workspace for at bekræfte at `my-cluster` er tilgængeligt.

*Hint:* `ml_client.compute.list()` returnerer alle compute-ressourcer.

In [5]:
# TODO: List alle compute-targets og print deres navn og type
# HINT: Iterér over ml_client.compute.list() og print compute.name og compute.type

for c in ml_client.compute.list():
    string = f"{c.name}: {c.type}"
    print(string)

GadeCompute: computeinstance
my-cluster: amlcompute


## 3. Forbered data til AutoML

AutoML kræver data som et Azure ML data asset. Vi bruger det eksisterende `ibm-churn-mltable:1` asset fra Dag 1, da MLTable er det anbefalede format til AutoML — det giver AutoML adgang til skema-information om kolonnerne.

AutoML bruger `Input`-objekter med typen `AssetTypes.MLTABLE` til at pege på training- og validationsdata.

> **Eksamen tip:** AutoML understøtter både `URI_FILE`/`URI_FOLDER` (som CSV) og `MLTABLE` som input. `MLTABLE` anbefales fordi det muliggør schema-validering og bedre integration med AutoML's featurization-pipeline. Valideringssplit kan enten ske automatisk (default: 10% stratified) eller du kan angive `validation_data` eksplicit.

**Opgave:** Opret et `Input`-objekt der peger på `ibm-churn-mltable:1` data-asset med typen `MLTABLE`.

*Hint:* `Input(type=AssetTypes.MLTABLE, path="azureml:ibm-churn-mltable:1")`

In [27]:
from azure.ai.ml import Input
from azure.ai.ml.constants import AssetTypes

# TODO: Opret et Input-objekt der peger på ibm-churn-mltable:1 med typen MLTABLE
training_data = Input(type=AssetTypes.MLTABLE, path="azureml:ibm-churn-mltable:1")

training_data

{'type': 'mltable', 'path': 'azureml:ibm-churn-mltable:1'}

## 4. Konfigurer AutoML klassifikationsjob

I SDK v2 konfigureres et AutoML klassifikationsjob med `automl.classification()`. De vigtigste parametre er:

| Parameter | Beskrivelse |
|-----------|-------------|
| `compute` | Compute-target navn (f.eks. `"my-cluster"`) |
| `experiment_name` | MLflow experiment-navn |
| `training_data` | `Input`-objekt med træningsdata |
| `target_column_name` | Navn på label-kolonnen |
| `primary_metric` | Metric der optimeres (f.eks. `"AUC_weighted"`) |
| `n_cross_validations` | Antal CV-folds (alternativ til validation_data) |

> **Eksamen tip:** `primary_metric` for klassifikation kan være: `accuracy`, `AUC_weighted`, `average_precision_score_weighted`, `norm_macro_recall`, `precision_score_weighted`. For **ubalancerede** datasæt (som attrition-data, der har ~84% 'No') anbefales `AUC_weighted` fremfor `accuracy`, da accuracy kan være vildledende.

**Opgave:** Konfigurer et AutoML klassifikationsjob med følgende indstillinger:
- `compute="my-cluster"`
- `experiment_name="attrition-automl"`
- `target_column_name="Attrition"`
- `primary_metric="AUC_weighted"`
- `n_cross_validations=3`

*Hint:* `from azure.ai.ml import automl` — brug derefter `automl.classification(...)`

In [28]:
from azure.ai.ml import automl

# TODO: Konfigurer et AutoML klassifikationsjob for attrition-datasættet
# Brug "my-cluster", sæt primary_metric til AUC_weighted, og brug 3-fold cross-validation

classification_job = automl.classification(
    training_data = training_data,
    target_column_name = "Attrition",
    primary_metric = "AUC_WEIGHTED",
    experiment_name = "automl-demo",
    n_cross_validations = 5,
    compute = "my-cluster"
)

print(classification_job)

compute: azureml:my-cluster
experiment_name: automl-demo
log_verbosity: info
n_cross_validations: 5
outputs: {}
primary_metric: auc_weighted
properties: {}
tags: {}
target_column_name: Attrition
task: classification
training: {}
training_data:
  path: azureml:ibm-churn-mltable:1
  type: mltable
type: automl



## 5. Sæt limits og featurization

AutoML-jobs kan løbe i timevis og koste mange penge hvis de ikke begrænses. Azure ML SDK v2 lader dig sætte grænser via `.set_limits()` og konfigurere feature engineering via `.set_featurization()`.

**`set_limits()`** — vigtige parametre:
- `timeout_minutes` — max total køretid for hele AutoML-jobbet
- `trial_timeout_minutes` — max tid per individuel model-trial
- `max_trials` — max antal model-kombinationer der afprøves
- `max_concurrent_trials` — parallelle trials (bør matche antal compute-noder)
- `enable_early_termination` — stop jobbet hvis ingen forbedring i X iterationer

**`set_featurization()`** — vigtige parametre:
- `mode` — `"auto"` (default), `"off"`, eller `"custom"`
  - `"auto"`: AutoML håndterer imputation, normalisering, encoding automatisk
  - `"off"`: Ingen automatisk feature engineering — data bruges som-er
  - `"custom"`: Du specificerer transformationer per kolonne

> **Eksamen tip:** `featurization="auto"` er default og inkluderer: missing value imputation, categorical encoding (one-hot/label), numeric scaling, og feature sweeping. Hvis dit data allerede er preprocessed (f.eks. fra et pipeline-trin), brug `"off"` for at undgå dobbelt transformation.

**Opgave:** Tilføj limits og featurization til dit `classification_job`:
- Limits: `timeout_minutes=20`, `trial_timeout_minutes=5`, `max_trials=8`, `max_concurrent_trials=2`, `enable_early_termination=True`
- Featurization: `mode="auto"`

*Hint:* Kald `.set_limits(...)` og `.set_featurization(...)` direkte på `classification_job`-objektet.

In [29]:
# Limits
classification_job.set_limits(
    timeout_minutes=20, 
    trial_timeout_minutes=5, 
    max_trials=8,
    max_concurrent_trials=2,
    enable_early_termination=True,
)

# Featurization
classification_job.set_featurization(
    mode="auto"
)

print("Limits og featurization sat.")
print(classification_job)

Limits og featurization sat.
compute: azureml:my-cluster
experiment_name: automl-demo
featurization:
  mode: auto
limits:
  enable_early_termination: true
  max_concurrent_trials: 2
  max_trials: 8
  timeout_minutes: 20
  trial_timeout_minutes: 5
log_verbosity: info
n_cross_validations: 5
outputs: {}
primary_metric: auc_weighted
properties: {}
tags: {}
target_column_name: Attrition
task: classification
training: {}
training_data:
  path: azureml:ibm-churn-mltable:1
  type: mltable
type: automl



## 6. Submit AutoML-jobbet

AutoML-jobs submittes på samme måde som command jobs — via `ml_client.jobs.create_or_update()`. Metoden returnerer et job-objekt med `name`, `status` og `studio_url`.

> **Eksamen tip:** AutoML-jobbet er et **parent job** der indeholder mange **child jobs** (en per model-trial). I Azure ML Studio kan du se alle child jobs under det overordnede AutoML-job og sammenligne metrics på tværs af dem. `ml_client.jobs.stream()` venter på det overordnede job — ikke de individuelle trials.

**Opgave:** Submit `classification_job` og print job-navn og Studio URL.

*Hint:* `returned_job = ml_client.jobs.create_or_update(classification_job)`

In [30]:
# TODO: Submit classification_job og print job-navn og Studio URL
returned_job = ml_client.jobs.create_or_update(classification_job)
print(f"Created job: {returned_job.name}")
print(f"Studio URL: {returned_job.studio_url}")

Created job: quirky_nerve_rd6x9cq0tg
Studio URL: https://ml.azure.com/runs/quirky_nerve_rd6x9cq0tg?wsid=/subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourcegroups/data-scientist-cert-rg/workspaces/data-scientist-cert&tid=fd728459-8712-40a7-973d-772eb8a1976d


In [31]:
# TODO: Stream job-logs mens du venter på completion
# ADVARSEL: AutoML-jobs kan tage 20-30+ minutter

ml_client.jobs.stream(returned_job.name)

RunId: quirky_nerve_rd6x9cq0tg
Web View: https://ml.azure.com/runs/quirky_nerve_rd6x9cq0tg?wsid=/subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourcegroups/data-scientist-cert-rg/workspaces/data-scientist-cert

Execution Summary
RunId: quirky_nerve_rd6x9cq0tg
Web View: https://ml.azure.com/runs/quirky_nerve_rd6x9cq0tg?wsid=/subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourcegroups/data-scientist-cert-rg/workspaces/data-scientist-cert



## 7. Hent det bedste run

Når AutoML-jobbet er færdigt, kan du hente det bedste child-run via `ml_client.jobs.list()` filtreret på parent job-navnet, eller via MLflow's `search_runs()` med experiment-filteret.

Den nemmeste metode i SDK v2 er at bruge `mlflow.search_runs()` og sortere på `primary_metric`. AutoML logger automatisk alle trials som MLflow runs under experiment-navnet.

> **Eksamen tip:** Det bedste AutoML-run kan identificeres på to måder:
> 1. Via `mlflow.search_runs(experiment_names=["attrition-automl"], order_by=["metrics.AUC_weighted DESC"])`
> 2. Via Azure ML Studio → Jobs → [dit AutoML job] → Best model summary
>
> Det bedste run har automatisk en tag `mlflow.runName` der starter med den valgte algoritme-navn.

**Opgave:** Hent alle runs fra `attrition-automl` eksperimentet og vis de 5 bedste sorteret på `AUC_weighted`.

*Hint:* `mlflow.search_runs(experiment_names=["attrition-automl"], order_by=["metrics.AUC_weighted DESC"], max_results=5)`

In [32]:
import mlflow

tracking_uri = ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri
mlflow.set_tracking_uri(tracking_uri)

all_runs = mlflow.search_runs()
all_runs.head()

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.val_accuracy,metrics.training_roc_auc,metrics.training_recall_score,metrics.training_accuracy_score,...,params.multi_class,params.random_state,params.n_jobs,params.class_weight,params.tol,tags.mlflow.rootRunId,tags.mlflow.runName,tags.mlflow.user,tags.estimator_class,tags.estimator_name
0,be8a072d-1a54-4364-8529-b41ef2ce6c61,9fa5c13e-d70b-4611-a9b1-3292a020db41,FINISHED,,2026-02-04 08:04:54.164000+00:00,2026-02-04 08:04:56.901000+00:00,NaN,NaN,NaN,NaN,...,None,None,None,None,None,be8a072d-1a54-4364-8529-b41ef2ce6c61,frosty_thread_c69kb6yd,Morten Gade,None,None
1,439cc161-fed5-4e1c-aa54-96541839d432,9fa5c13e-d70b-4611-a9b1-3292a020db41,FINISHED,,2026-02-04 08:08:48.436000+00:00,2026-02-04 08:08:49.204000+00:00,NaN,NaN,NaN,NaN,...,None,None,None,None,None,439cc161-fed5-4e1c-aa54-96541839d432,blue_tail_dpbj56v9,Morten Gade,None,None
2,99a7c882-2f19-46bc-adbd-241a3f2ea5dd,9fa5c13e-d70b-4611-a9b1-3292a020db41,FINISHED,,2026-02-04 08:10:09.843000+00:00,2026-02-04 08:10:10.903000+00:00,NaN,NaN,NaN,NaN,...,None,None,None,None,None,99a7c882-2f19-46bc-adbd-241a3f2ea5dd,silver_brick_j1bqspyk,Morten Gade,None,None
3,84a8d15b-2637-4161-9c0e-fe846e96654b,9fa5c13e-d70b-4611-a9b1-3292a020db41,FINISHED,,2026-02-04 08:13:09.554000+00:00,2026-02-04 08:13:32.284000+00:00,0.825472,0.73044,0.844287,0.844287,...,deprecated,None,None,None,0.0001,84a8d15b-2637-4161-9c0e-fe846e96654b,gifted_tiger_13bt8lhk,Morten Gade,sklearn.linear_model._logistic.LogisticRegression,LogisticRegression
4,e4f20359-c9e8-41d8-967a-7de250be19db,9fa5c13e-d70b-4611-a9b1-3292a020db41,FINISHED,,2026-02-04 08:19:09.152000+00:00,2026-02-04 08:19:12.726000+00:00,NaN,0.73044,0.844287,0.844287,...,deprecated,None,None,None,0.0001,e4f20359-c9e8-41d8-967a-7de250be19db,joyful_nutmeg_fldl6w7k,Morten Gade,sklearn.linear_model._logistic.LogisticRegression,LogisticRegression


In [43]:
automl_runs = mlflow.search_runs(experiment_ids=['5401f9ed-acd2-4b6e-9c59-e255ff9a8fba'])
automl_runs = automl_runs[(automl_runs['status']=='FINISHED')][['run_id', 'experiment_id', 'metrics.AUC_weighted']]
automl_runs.sort_values(by="metrics.AUC_weighted", ascending=False)

,run_id,experiment_id,metrics.AUC_weighted
2,quirky_nerve_rd6x9cq0tg,5401f9ed-acd2-4b6e-9c59-e255ff9a8fba,0.816534
14,quirky_nerve_rd6x9cq0tg_6,5401f9ed-acd2-4b6e-9c59-e255ff9a8fba,0.816534
15,quirky_nerve_rd6x9cq0tg_7,5401f9ed-acd2-4b6e-9c59-e255ff9a8fba,0.810545
11,quirky_nerve_rd6x9cq0tg_4,5401f9ed-acd2-4b6e-9c59-e255ff9a8fba,0.803025
6,quirky_nerve_rd6x9cq0tg_0,5401f9ed-acd2-4b6e-9c59-e255ff9a8fba,0.799763
7,quirky_nerve_rd6x9cq0tg_1,5401f9ed-acd2-4b6e-9c59-e255ff9a8fba,0.786801
8,quirky_nerve_rd6x9cq0tg_2,5401f9ed-acd2-4b6e-9c59-e255ff9a8fba,0.773987
12,quirky_nerve_rd6x9cq0tg_5,5401f9ed-acd2-4b6e-9c59-e255ff9a8fba,0.768671
10,quirky_nerve_rd6x9cq0tg_3,5401f9ed-acd2-4b6e-9c59-e255ff9a8fba,0.749652
3,quirky_nerve_rd6x9cq0tg_setup,5401f9ed-acd2-4b6e-9c59-e255ff9a8fba,NaN


## 8. Hent og inspicér den bedste model

AutoML-modeller kan hentes via MLflow's `mlflow.pyfunc.load_model()` — dette virker for alle AutoML-modeller uanset hvilken algoritme der vandt. Den returnerede model er et `PythonModel` interface med en `.predict()` metode.

Alternativt kan du hente modellen som et Azure ML `Model`-objekt og registrere den i Model Registry (se Dag 7).

> **Eksamen tip:** AutoML logger altid modellen som et MLflow run-artefakt under `artifact_path="outputs/mlflow-model"`. Stien til artefaktet er `runs:/<run_id>/outputs/mlflow-model`. Du kan bruge `mlflow.pyfunc.load_model(f"runs:/{best_run_id}/outputs/mlflow-model")` til at indlæse den bedste model direkte.

**Opgave:** Hent run_id for det bedste run og indlæs modellen med `mlflow.pyfunc.load_model()`.

*Hint:* `best_run_id = best_runs.iloc[0]["run_id"]` og derefter `mlflow.pyfunc.load_model(f"runs:/{best_run_id}/outputs/mlflow-model")`

In [ ]:
import pandas as pd

# TODO: Hent run_id for det bedste run og indlæs modellen
# HINT: AutoML gemmer modellen under artifact path "outputs/mlflow-model"
best_run_id = automl_runs.sort_values(by="metrics.AUC_weighted", ascending=False).iloc[1,0]
# best_run_id = 'quirky_nerve_rd6x9cq0tg_6'

# Det her virkede ikke:
mlflow.pyfunc.load_model(f"runs:/{best_run_id}/outputs/mlflow-model") 
# Fik fejlen "ModuleNotFoundError: No module named 'azureml.training'"

# Det her virkede heller ikke:
# mlflow.sklearn.load_model(f"runs:/{best_run_id}/outputs/mlflow-model")

# I stedet registrerer jeg den bedste model

In [48]:
from azure.ai.ml.entities import Model

best_model = Model(
    path=f"azureml://jobs/{best_run_id}/outputs/artifacts/outputs/mlflow-model",
    name="attrition-automl-best",
    type=AssetTypes.MLFLOW_MODEL,
    description="Best AutoML classification model for attrition"
)

ml_client.models.create_or_update(best_model)

Model({'job_name': 'quirky_nerve_rd6x9cq0tg_6', 'intellectual_property': None, 'system_metadata': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'attrition-automl-best', 'description': 'Best AutoML classification model for attrition', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourceGroups/data-scientist-cert-rg/providers/Microsoft.MachineLearningServices/workspaces/data-scientist-cert/models/attrition-automl-best/versions/1', 'Resource__source_path': '', 'base_path': '/Users/gade/Knowit/DP100/notebooks', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x1517b1250>, 'serialize': <msrest.serialization.Serializer object at 0x1332c02c0>, 'version': '1', 'latest_version': None, 'path': 'azureml://subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourceGroups/data-scientist-cert-rg/workspaces/data-scientist-cert/datastores/workspaceartifa

In [58]:
from mlflow.artifacts import download_artifacts
local_path = download_artifacts(run_id=best_run_id, artifact_path="outputs/mlflow-model")
print(local_path)

/var/folders/ww/p8sz7czn2wd3cxvwdtxtc9zr0000gn/T/tmp6j3ele2u/outputs/mlflow-model


In [62]:
# Se hvilken algoritme AutoML valgte
!cat /var/folders/ww/p8sz7czn2wd3cxvwdtxtc9zr0000gn/T/tmp6j3ele2u/outputs/mlflow-model/MLmodel

# Se dependencies modellen kræver
!cat /var/folders/ww/p8sz7czn2wd3cxvwdtxtc9zr0000gn/T/tmp6j3ele2u/outputs/mlflow-model/conda.yaml

artifact_path: outputs/mlflow-model
flavors:
  python_function:
    env:
      conda: conda.yaml
      virtualenv: python_env.yaml
    loader_module: mlflow.sklearn
    model_path: model.pkl
    predict_fn: predict
    python_version: 3.10.19
  sklearn:
    code: null
    pickled_model: model.pkl
    serialization_format: pickle
    sklearn_version: 1.5.1
metadata:
  azureml.base_image: mcr.microsoft.com/azureml/curated/ai-ml-automl:38
  azureml.engine: automl
mlflow_version: 2.15.1
model_size_bytes: 859052
model_uuid: 228f97abec4748489d739e1c074aacf3
run_id: quirky_nerve_rd6x9cq0tg_6
signature:
  inputs: '[{"type": "long", "name": "Age", "required": true}, {"type": "string",
    "name": "BusinessTravel", "required": true}, {"type": "long", "name": "DailyRate",
    "required": true}, {"type": "string", "name": "Department", "required": true},
    {"type": "long", "name": "DistanceFromHome", "required": true}, {"type": "long",
    "name": "Education", "required": true}, {"type": "string

In [59]:
mlflow.sklearn.load_model(local_path)

ModuleNotFoundError: No module named 'azureml.training'

## 9. Kør inferens med AutoML-modellen

Den indlæste MLflow `pyfunc`-model kan bruges direkte til at lave forudsigelser på nye data. Input skal have samme kolonner som træningsdataet (minus label-kolonnen).

> **Eksamen tip:** AutoML-modeller pakket som `mlflow.pyfunc` accepterer input som en pandas DataFrame. Kolonnenavne skal matche præcis (inkl. store/små bogstaver). AutoML's featurization sker **inden** modellen — den gemte MLflow-model indeholder hele pipeline'en inkl. preprocessing, så du sender rådata direkte til `.predict()`.

**Opgave:** Indlæs de første 5 rækker fra `../data/churn-raw.csv`, fjern `Attrition`-kolonnen og kald `best_model.predict()` på dem.

*Hint:* `df = pd.read_csv("../data/churn-raw.csv")` og `sample = df.drop(columns=["Attrition"]).head(5)`

In [ ]:
# TODO: Indlæs churn-data, fjern Attrition-kolonnen, og kør predict() på de første 5 rækker
df = pd.read_csv("../data/churn-raw.csv")
sample = df.drop(columns=['Attrition']).head(5)



## 10. AutoML Regression (bonus)

AutoML understøtter ikke kun klassifikation — du kan også bruge det til regression med `automl.regression()`. Konfigurationen er næsten identisk, men `primary_metric` er anderledes.

Tilgængelige `primary_metric` for **regression**:
- `normalized_root_mean_squared_error` (default)
- `r2_score`
- `normalized_mean_absolute_error`
- `spearman_correlation`

> **Eksamen tip:** `normalized_root_mean_squared_error` er default for regression og **minimeres** (lavere er bedre). For klassifikation **maksimeres** `AUC_weighted` (højere er bedre). AutoML vælger automatisk den rigtige retning baseret på metric-typen — du behøver ikke angive om det er min/max.

**Opgave:** Konfigurer et AutoML regressionsjob der forudsiger `MonthlyIncome` fra de øvrige kolonner. Brug `r2_score` som `primary_metric` og sæt strenge limits (max 5 trials, 15 minutters timeout) så det ikke koster for meget.

*Hint:* `automl.regression(target_column_name="MonthlyIncome", primary_metric="r2_score", ...)`

In [64]:
# TODO: Konfigurer et AutoML regressionsjob der forudsiger MonthlyIncome
# Brug r2_score som primary_metric og sæt strenge limits (max 5 trials, 15 min timeout)
# Submit jobbet og print Studio URL

from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import Input

regression_input = Input(
    type=AssetTypes.MLTABLE, path="azureml:ibm-churn-mltable:1"
)

# Define job
from azure.ai.ml.automl import regression
regression_job = regression(
    training_data=regression_input,
    target_column_name="MonthlyIncome",
    primary_metric="r2_score",
    experiment_name = "automl-regression-demo",
    n_cross_validations = 5,
    compute = "my-cluster"
)

# Limits
regression_job.set_limits(
    timeout_minutes=15, 
    trial_timeout_minutes=5, 
    max_trials=4,
    max_concurrent_trials=2,
    enable_early_termination=True,
)

# Featurization
regression_job.set_featurization(
    mode="auto"
)

# Run job
returned_reg_job = ml_client.jobs.create_or_update(regression_job)
print(f"Job Name: {returned_reg_job.name}")
print(f"Studio URL: {returned_reg_job.studio_url}")

Job Name: magenta_bear_krt582z28m
Studio URL: https://ml.azure.com/runs/magenta_bear_krt582z28m?wsid=/subscriptions/9bf05e2c-7ab6-4705-b7af-45b13a47bfe2/resourcegroups/data-scientist-cert-rg/workspaces/data-scientist-cert&tid=fd728459-8712-40a7-973d-772eb8a1976d


## Refleksion

Besvar disse spørgsmål med dine egne ord:

1. Hvad er fordelen ved at bruge `AUC_weighted` frem for `accuracy` som `primary_metric` på attrition-datasættet?

Kan læse mig til, at AUC_weighted er bedre til at håndtere imbalanced data. Det passer til datasættet, eftersom der er en overvægt af observationer, hvor den ansatte *ikke* stopper. 

Accuracy er måske farlig fordi man kan risikere en "dum model", der altid gætter på majoritetsklassen (i det her tilfælde 'Attrition' = 0).

2. Hvad gør `featurization="auto"` konkret — hvad sker der med kolonner som `Gender` og `Department`?

Tror der sker følgende væsentlige ting:
- auto impute af missing values
- auto encoding af categorical values

Så hvis alt fungerer, så bliver gender til én række ved navn Gender_Male, som enten er 0 eller 1. Den skal egentlig one-hot encodes, kolonnenavnet skal blot stadig være informativt ('Male' i stedet for 'Gender' fx).

'Department' skal der laves dummy variables på, fordi antal unikke departmenets er større end 2.

3. Hvad er forskellen på `timeout_minutes` og `trial_timeout_minutes` i `set_limits()`?

timeout_minutes(): overordnet timeout
trial_timeout_minutes(): mere granuleret timeout, på trial-niveau

4. Hvordan ved AutoML hvilken retning (min/max) den skal optimere `primary_metric` i?

Godt spørgsmål... tænker man kan sætte det, men ellers bliver det automatisk udledt baseret på typen af opgave (regression/classification) samt navnet på metrikken.
Fx hvis type="classification" og metric="accuracy" --> "maximize".
Eller hvis type="regression" og metric="rmse" --> "minimize".

5. Hvornår ville du vælge et manuelt `command()` job med `src/train.py` frem for AutoML?

Det kan der være mange årsager til. 
AutoML er måske fint hvis alt handler om forudsigelse.

Hvis vi ønsker at lave kausal inferens, handler det jo netop om at opstille en model meget bevidst, med udgangspunkt i teori vedr. relationerne mellem X-variable og y-variabel. 

Der kan også være teoretisk funderede grunde til at benytte sig af bestemte statistiske modeller, bestemt featurization, bestemt metric-set osv.


## Nøglepunkter (DP-100 eksamen)

- **`automl.classification()` / `automl.regression()`** — SDK v2 funktioner i `azure.ai.ml.automl` til at konfigurere AutoML-jobs. Begge returnerer et job-objekt der submittes med `ml_client.jobs.create_or_update()`.
- **`primary_metric`** — Klassifikation: `AUC_weighted`, `accuracy`, `average_precision_score_weighted` m.fl. Regression: `normalized_root_mean_squared_error`, `r2_score` m.fl. AutoML håndterer min/max automatisk.
- **`featurization`** — `"auto"` (default): AutoML håndterer imputation, encoding, scaling. `"off"`: ingen preprocessing. `"custom"`: bruger-definerede transformationer per kolonne.
- **`set_limits()`** — Kontrollér ressourceforbrug: `timeout_minutes` (total), `trial_timeout_minutes` (per trial), `max_trials`, `max_concurrent_trials`, `enable_early_termination`.
- **AutoML og MLflow** — Hvert AutoML-trial er et MLflow child-run. Brug `mlflow.search_runs(experiment_names=[...], order_by=[...])` til at finde det bedste run programmatisk.
- **Model-format** — AutoML gemmer modeller som `mlflow.pyfunc` under `outputs/mlflow-model`. Indlæs med `mlflow.pyfunc.load_model(f"runs:/{run_id}/outputs/mlflow-model")`. Modellen indeholder hele preprocessing-pipeline.
- **`n_cross_validations`** — Bruges som alternativ til explicit `validation_data`. Anbefalet for mindre datasæt (< 20.000 rækker) for at reducere variance i metric-estimering.
- **Fase 2 eksamenssektionen** (Explore data & Experiments, 20-25%) — AutoML er centralt her og kombinerer godt med MLflow-tracking fra Dag 3.